# 06 — Full Training Pipeline with Flax NNX + Optax

This notebook shows production-quality training with:
- `nnx.Optimizer` wrapping an Optax optimizer
- Gradient clipping
- Train/val split with metrics logging
- Checkpointing with Orbax
- `nnx.jit` for the full train/eval loop

In [ ]:
import sys; sys.path.insert(0, '..')
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.rnn_jax import VanillaRNNModel, LowRankRNNModel
from src.utils import make_sinusoid_task
from src.train import mse_loss, make_train_step, make_eval_step, make_optimizer

## 6.1 Data: Train / Val Split

In [ ]:
N_TRAIN, N_VAL, SEQ_LEN = 800, 200, 100
all_data = make_sinusoid_task(N_TRAIN + N_VAL, SEQ_LEN, key=jax.random.PRNGKey(0))

# Batch is axis 0 (batch-first convention)
train_data = {k: v[:N_TRAIN] for k, v in all_data.items()}
val_data   = {k: v[N_TRAIN:] for k, v in all_data.items()}
print('Train sequences:', train_data['inputs'].shape[0])
print('Val   sequences:', val_data['inputs'].shape[0])
print('Data shape (B, T, I):', train_data['inputs'].shape)

## 6.2 Model + Optimizer Setup

In [ ]:
model = LowRankRNNModel(
    input_size=1,
    hidden_size=64,
    rank=4,
    output_size=1,
    rngs=nnx.Rngs(42),
)

# Optax optimizer with gradient clipping
tx = make_optimizer(lr=3e-3, max_grad_norm=1.0)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)

# Count params
_, state = nnx.split(model)
n_params = sum(v.size for v in jax.tree.leaves(state))
print(f'Model params: {n_params:,}')

## 6.3 JIT-Compiled Train and Eval Steps

In [ ]:
def loss_fn(model, batch):
    return mse_loss(model(batch['inputs']), batch['targets'])

train_step = make_train_step(loss_fn)
eval_step  = make_eval_step(loss_fn)

## 6.4 Training Loop with Metrics

In [ ]:
N_EPOCHS = 20
BATCH_SIZE = 64
key = jax.random.PRNGKey(0)

train_losses, val_losses = [], []

for epoch in range(N_EPOCHS):
    # ---------- train ----------
    key, sk = jax.random.split(key)
    idx = jax.random.permutation(sk, N_TRAIN)
    ep_loss = 0.0
    n_batches = 0
    for i in range(0, N_TRAIN, BATCH_SIZE):
        batch_idx = idx[i:i+BATCH_SIZE]
        batch = {k: v[batch_idx] for k, v in train_data.items()}  # batch on axis 0
        loss = train_step(model, optimizer, batch)
        ep_loss += float(loss)
        n_batches += 1
    train_losses.append(ep_loss / n_batches)

    # ---------- validate ----------
    val_loss = float(eval_step(model, val_data))
    val_losses.append(val_loss)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS}  train={train_losses[-1]:.5f}  val={val_losses[-1]:.5f}')

print('Done.')

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(train_losses, label='train')
plt.semilogy(val_losses, '--', label='val')
plt.xlabel('Epoch'); plt.ylabel('MSE loss')
plt.title('Train vs Validation loss')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 6.5 Checkpointing with Orbax

In [ ]:
import orbax.checkpoint as ocp

# Orbax requires an absolute path
ckpt_dir = (Path('..') / 'checkpoints' / 'lowrank_rnn').resolve()
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Save
_, state = nnx.split(model)
checkpointer = ocp.StandardCheckpointer()
checkpointer.save(ckpt_dir / 'step_final', state)
print('Checkpoint saved to', ckpt_dir / 'step_final')

# Restore
model_restored = LowRankRNNModel(1, 64, rank=4, output_size=1, rngs=nnx.Rngs(99))
graphdef, abstract_state = nnx.split(model_restored)
restored_state = checkpointer.restore(ckpt_dir / 'step_final', abstract_state)
model_restored = nnx.merge(graphdef, restored_state)

# Verify — xs_test is batch-first (B=4, T=10, I=1)
xs_test = jnp.ones((4, 10, 1))
diff = jnp.max(jnp.abs(model(xs_test) - model_restored(xs_test)))
print(f'Max diff after restore: {float(diff):.2e}  (should be ~0)')

## 6.6 Learning Rate Schedules

In [ ]:
# Cosine annealing schedule with warmup
n_steps_total = N_EPOCHS * (N_TRAIN // BATCH_SIZE)

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=3e-3,
    warmup_steps=n_steps_total // 10,
    decay_steps=n_steps_total,
    end_value=1e-5,
)

lrs = [float(schedule(i)) for i in range(n_steps_total)]
plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel('Step'); plt.ylabel('Learning rate')
plt.title('Warmup + cosine decay schedule')
plt.grid(alpha=0.3); plt.show()